# 第 30 章 · 迁移与交付：程序、数据和激活记录一起核对

升级不是解压新代码：新程序要能找到 helper、读懂迁移后的数据、通过语义检查，最后才能切换 active 指针。数据迁移完成与新版本已生效是两个不同时间点。

本章使用真实文件、SQLite、tar 和本地子进程，但所有包都在本次实验目录生成，仅包含诊断小程序。需要 POSIX（macOS/Linux）；没有安装器、联网下载、签名发布、真实 Mini Codex 最终包或崩溃持久性验收。

**运行范围：局部机制实验。** 按顺序执行，每个实验都通过真实函数、文件或队列产生结果，再用断言检查。修改参数后重跑该实验，必要时先重启内核并执行全部。所有合成输入都只证明本地机制；本章没有累计检查点，真实服务与设备仍待验证。

本章使用原作者的设计与局部原型；下面明确写出的简化是教材选择，不是 Codex 原样实现。固定源码：`c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee`。

In [1]:
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'code/mini_codex/__init__.py').is_file())
sys.path.insert(0, str(ROOT/'code'))
from mini_codex.learning import load_example, show_source, fresh_run_dir, run_python, run_chapter

## 1 · 先 preview，后导入；身份阻止重复历史

preview 只读输入，不创建目标数据库。真正导入写入来源摘要和稳定 ID；同 source_id 换内容会拒绝，防止“同一历史”被静默替换。修改任务 title 时必须用新 source_id 或明确迁移策略。

In [2]:
import json
import os
assert os.name == 'posix', '本章使用 POSIX fcntl，Windows 保留待验证'
m = load_example('30', 'upgrade_lab')
run_dir = fresh_run_dir('ch30')
source = run_dir/'source.json'
source_value = {'format_version': 1, 'source_id': 'notebook-export-1',
    'tasks': [{'id': 't1', 'title': '修复筛选', 'done': True}, {'id': 't2', 'title': '保留历史', 'done': False}],
    'threads': [{'id': 'thread1', 'messages': [{'role': 'user', 'text': '受控历史输入'}]}],
    'memory': [{'id': 'memory1', 'text': '任务 ID 保持稳定'}], 'config': {'project': 'task-list'}}
m.atomic_json(source, source_value)
target = run_dir/'imported'/'history.sqlite'
preview = m.preview(source, target)
assert preview['action'] == 'import' and not target.parent.exists()
m.import_source(source, target)
assert m.import_source(source, target)['action'] == 'already_imported'
before = m.inspect_db(target)
assert before['schema'] == 1 and len(before['tasks']) == 2
changed_source = run_dir/'changed-source.json'
m.atomic_json(changed_source, {**source_value, 'config': {'project': 'changed'}})
try:
    m.preview(changed_source, target)
except m.Refused as error:
    assert 'different content' in str(error)
else:
    raise AssertionError('相同来源 ID 不能静默换内容')
print({'preview_counts': preview['counts'], 'observed_schema': before['schema']})

{'preview_counts': {'tasks': 2, 'threads': 1, 'messages': 1, 'memory': 1}, 'observed_schema': 1}


## 2 · 用真实包启动诊断进程，再切换版本

包的 manifest、实际程序版本、helper 协议和数据 schema 要一致。这里真实启动生成的诊断程序，读取数据库观察版本；它不执行 Agent 任务。升级由 done 布尔值转换成 todo/done 状态，语义视图必须相同。

In [3]:
v1, v2 = run_dir/'diagnostic-v1.tar.gz', run_dir/'diagnostic-v2.tar.gz'
h1, h2 = m.build_package(v1, '1.0.0'), m.build_package(v2, '2.0.0')
def installed(name):
    root = run_dir/name
    m.bootstrap(root, source, v1, h1)
    return root
root = installed('normal')
assert m.run_installed(root)['program_version'] == '1.0.0'
updated = m.upgrade(root, v2, h2)
observed = m.run_installed(root)
assert updated['status'] == 'updated'
assert observed['program_version'] == '2.0.0' and observed['schema'] == 2
active = m.read_json(root/'active.json')
after = m.inspect_db(root/active['database'])
assert after['tasks'] == before['tasks'] and after['messages'] == before['messages']
print({'diagnostic_process': observed, 'active': active})

{'diagnostic_process': {'program_version': '2.0.0', 'schema': 2, 'task_count': 2, 'helper_protocol': 1}, 'active': {'database': 'data/4c3b1ce90bef451da397b0c40537b44d.sqlite', 'package': 'packages/4c3b1ce90bef451da397b0c40537b44d', 'schema': 2, 'tx': '4c3b1ce90bef451da397b0c40537b44d', 'version': '2.0.0'}}


## 3 · 语义失败或取消都不激活候选

SQLite integrity_check 通过，不代表业务记录没有丢失。`corrupt_candidate=True` 故意删除一条记录，语义 gate 应拒绝。取消发生在准备完成后、active 切换前；旧版本仍应实际可运行。

In [4]:
corrupt_root = installed('corrupt-candidate')
try:
    m.upgrade(corrupt_root, v2, h2, corrupt_candidate=True)
except m.Refused as error:
    assert str(error) == 'candidate data semantic verification failed'
else:
    raise AssertionError('丢记录的候选不应激活')
assert m.run_installed(corrupt_root)['program_version'] == '1.0.0'
cancel_root = installed('cancelled')
cancelled = m.upgrade(cancel_root, v2, h2, cancel=True)
assert cancelled['status'] == 'cancelled'
assert m.run_installed(cancel_root)['program_version'] == '1.0.0'
recovery = m.recover(corrupt_root)
assert recovery and recovery[0]['decision'] == 'ABORTED'
print({'semantic_failure_active': '1.0.0', 'cancel_active': '1.0.0', 'failed_preparation_recovery': recovery})

{'semantic_failure_active': '1.0.0', 'cancel_active': '1.0.0', 'failed_preparation_recovery': [{'tx': '44384226938148feaebb1db4b1e3e2dc', 'decision': 'ABORTED'}]}


## 4 · 激活后丢回执，恢复先核对磁盘

这里使用明确的 Python 异常故障注入，**不是杀进程或断电实验**。异常出现在 active 已切换、COMMITTED 日志未写入之间；恢复必须读取 active、候选摘要并实际启动诊断程序，再确认 COMMITTED。

In [5]:
lost_root = installed('lost-activation-ack')
try:
    m.upgrade(lost_root, v2, h2, fault='fail:activated_before_ack')
except m.Refused as error:
    assert str(error) == 'injected failure: activated_before_ack'
else:
    raise AssertionError('故障注入未发生')
assert m.read_json(lost_root/'active.json')['version'] == '2.0.0'
try:
    m.run_installed(lost_root, write='不能在恢复前写入')
except m.Refused as error:
    assert 'recover activated transaction' in str(error)
else:
    raise AssertionError('未恢复事务不能接纳新写入')
decisions = m.recover(lost_root)
assert decisions and decisions[0]['decision'] == 'COMMITTED'
assert m.run_installed(lost_root)['schema'] == 2
print({'recovery': decisions, 'observed_version': m.run_installed(lost_root)['program_version']})

{'recovery': [{'tx': '139d9a0c32fa4bfa89fcc457bf4e5fbe', 'decision': 'COMMITTED'}], 'observed_version': '2.0.0'}


## 5 · 回滚不能抹掉升级后的用户写入

没有新写入时，可回切到保留的旧代。新版本已有真实用户写入后，自动回滚会丢数据，必须拒绝；后续需要专门的兼容迁移或人工处理。

In [6]:
rolled_back = m.rollback(root, updated['tx'])
assert rolled_back['status'] == 'rolled_back'
assert m.run_installed(root)['program_version'] == '1.0.0'
written = m.run_installed(lost_root, write='升级之后的新任务')
assert written['task_count'] == 3
active_lost = m.read_json(lost_root/'active.json')
try:
    m.rollback(lost_root, active_lost['tx'])
except m.Refused as error:
    assert 'post-upgrade user writes exist' in str(error)
else:
    raise AssertionError('自动回滚不能丢掉新写入')
assert m.run_installed(lost_root)['task_count'] == 3
print({'unchanged_generation_rollback': 'accepted', 'post_write_rollback': 'refused', 'retained_tasks': 3})

{'unchanged_generation_rollback': 'accepted', 'post_write_rollback': 'refused', 'retained_tasks': 3}


## 源码对照与接回主线

以下为固定提交的已定位源码入口。阅读源码能解释设计边界，不等于执行过其测试；本章 Python 原型与这些 Rust 接口不是同一实现。

- [codex-rs/exec-server/src/environment/discovery_v2.rs#L10-L55](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/exec-server/src/environment/discovery_v2.rs#L10-L55)
- [codex-rs/external-agent-migration/src/sessions/ledger.rs#L125-L204](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/external-agent-migration/src/sessions/ledger.rs#L125-L204)
- [codex-rs/external-agent-migration/src/service.rs#L573-L619](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/external-agent-migration/src/service.rs#L573-L619)

**接回条件：** 将局部机制接入同一 Dispatcher、Scheduler、History 和受保护验证器，再执行原合同。目录、断言和演示输出都不能替代累计集成证据。